# Лабораторная работа 3: Бинарная классификация
## Вариант: Breast Cancer Dataset (scikit-learn built-in)
### Сложность: Rare

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, roc_auc_score, roc_curve)

# Настройка визуализации
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. Загрузка данных

In [ ]:
# Загрузка датасета Breast Cancer
data = datasets.load_breast_cancer()

X = pd.DataFrame(data["data"], columns=data["feature_names"])
y = data["target"]

# Создание полного датафрейма
df = X.copy()
df['target'] = y

print("=" * 50)
print("ОПИСАНИЕ БАЗЫ ДАННЫХ")
print("=" * 50)
print(f"\nНазвание: {data.DESCR.split('\n')[0]}")
print(f"\nЦелевая переменная (target):")
print(f"  0 - злокачественная опухоль (malignant)")
print(f"  1 - доброкачественная опухоль (benign)")
print(f"\nПризнаки ({len(data['feature_names'])} штук): характеристики ядер клеток")
for i, feat in enumerate(data['feature_names'][:10], 1):
    print(f"  {i}. {feat}")
print(f"  ... и еще {len(data['feature_names'])-10} признаков")

print("\n" + "=" * 50)
print(f"Размер датасета: {df.shape[0]} строк x {df.shape[1]} столбцов")
print(f"Размер в памяти: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("=" * 50)

## 2. Исследовательский анализ данных (EDA)

In [ ]:
# Статистика для интервальных переменных
print("=" * 60)
print("СТАТИСТИКА ПО ИНТЕРВАЛЬНЫМ ПЕРЕМЕННЫМ")
print("=" * 60)

numeric_stats = df.drop(columns='target').describe(percentiles=[0.25, 0.5, 0.75]).T
numeric_stats = numeric_stats[['count', 'mean', '50%', 'min', '25%', '75%', 'max']]
numeric_stats.columns = ['Количество', 'Среднее', 'Медиана', 'Мин', '25%', '75%', 'Макс']

print(numeric_stats.to_string())
print("=" * 60)

In [ ]:
# Анализ целевой переменной
print("=" * 60)
print("РАСПРЕДЕЛЕНИЕ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ")
print("=" * 60)
target_counts = df['target'].value_counts()
target_percent = df['target'].value_counts(normalize=True) * 100

print(f"\nМода класса: {df['target'].mode()[0]}")
print(f"Количество моды: {target_counts[df['target'].mode()[0]]}")

print("\nРаспределение классов:")
for val, count in target_counts.items():
    class_name = "Доброкачественная" if val == 1 else "Злокачественная"
    print(f"  {class_name} ({val}): {count} ({target_percent[val]:.1f}%)")

print("=" * 60)

In [ ]:
# Проверка пропусков
print("=" * 60)
print("АНАЛИЗ ПРОПУСКОВ")
print("=" * 60)

missing = df.isnull().sum()
if missing.sum() == 0:
    print("\nПропусков не обнаружено!")
else:
    print("\nПропуски по столбцам:")
    print(missing[missing > 0])
print("=" * 60)

In [ ]:
# Анализ выбросов (IQR метод)
print("=" * 60)
print("АНАЛИЗ ВЫБРОСОВ")
print("=" * 60)

outliers_info = {}
for col in X.columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    outliers_info[col] = {
        'count': len(outliers),
        'percent': len(outliers) / len(df) * 100,
        'lower': lower,
        'upper': upper
    }

print(f"\nПризнаки с наибольшим количеством выбросов:")
sorted_outliers = sorted(outliers_info.items(), key=lambda x: x[1]['count'], reverse=True)
for col, info in sorted_outliers[:10]:
    print(f"  {col}: {info['count']} выбросов ({info['percent']:.1f}%)")

print("\nКатегориальные переменные отсутствуют - все признаки числовые.")
print("=" * 60)

In [ ]:
# Визуализация распределения признаков
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Распределение основных признаков', fontsize=14)

features_to_plot = ['mean radius', 'mean texture', 'mean perimeter', 
                    'mean area', 'mean smoothness', 'mean concavity']

for idx, feat in enumerate(features_to_plot):
    ax = axes[idx // 3, idx % 3]
    for target_val in [0, 1]:
        label = 'Злокачественная' if target_val == 0 else 'Доброкачественная'
        ax.hist(df[df['target'] == target_val][feat], bins=30, alpha=0.6, label=label)
    ax.set_xlabel(feat)
    ax.set_ylabel('Количество')
    ax.legend()

plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab3/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Корреляционная матрица
plt.figure(figsize=(14, 12))
correlation = df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, cmap='coolwarm', center=0,
            annot=False, fmt='.2f', cbar_kws={'label': 'Коэффициент корреляции'})
plt.title('Корреляционная матрица признаков', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab3/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Признаки с высокой корреляцией с целевой переменной
target_corr = correlation['target'].abs().sort_values(ascending=False)
print("\n" + "=" * 60)
print("ТОП-10 ПРИЗНАКОВ ПО КОРРЕЛЯЦИИ С ЦЕЛЕВОЙ ПЕРЕМЕННОЙ")
print("=" * 60)
for i, (feat, corr) in enumerate(target_corr[1:11].items(), 1):
    print(f"{i:2d}. {feat:25s}: {corr:.4f}")
print("=" * 60)

## 3. Подготовка данных

In [ ]:
# Разделение на train и test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("=" * 60)
print("РАЗДЕЛЕНИЕ ДАННЫХ")
print("=" * 60)
print(f"\nTrain set: {X_train.shape[0]} образцов ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} образцов ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nРаспределение классов в train:")
print(f"  0: {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"  1: {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")

print(f"\nРаспределение классов в test:")
print(f"  0: {(y_test==0).sum()} ({(y_test==0).sum()/len(y_test)*100:.1f}%)")
print(f"  1: {(y_test==1).sum()} ({(y_test==1).sum()/len(y_test)*100:.1f}%)")
print("=" * 60)

In [ ]:
# Нормализация данных (важно для KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nДанные нормализованы (StandardScaler: mean=0, std=1)")
print("Статистика после нормализации (train):")
print(f"  Среднее: {X_train_scaled.mean():.6f}")
print(f"  Стандартное отклонение: {X_train_scaled.std():.6f}")

## 4. Построение моделей классификации

In [ ]:
# Функция для оценки модели
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Обучает модель и возвращает метрики"""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    if y_pred_proba is not None:
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    else:
        roc_auc = None
        fpr, tpr = None, None
    
    return {
        'model': model,
        'name': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'roc_auc': roc_auc,
        'fpr': fpr,
        'tpr': tpr,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }

print("Функция оценки моделей создана")

In [ ]:
# Модель 1: KNN
print("=" * 60)
print("МОДЕЛЬ 1: K-Nearest Neighbors (KNN)")
print("=" * 60)

knn = KNeighborsClassifier(n_neighbors=5)
knn_results = evaluate_model(knn, X_train_scaled, X_test_scaled, y_train, y_test, "KNN")

print(f"\nAccuracy (A): {knn_results['accuracy']:.4f}")
print(f"Precision (P): {knn_results['precision']:.4f}")
print(f"Recall (R): {knn_results['recall']:.4f}")
print(f"F1-score: {knn_results['f1']:.4f}")
print(f"ROC-AUC: {knn_results['roc_auc']:.4f}")

print("\nConfusion Matrix:")
print("                Прогноз")
print("                0    1")
print(f"Факт    0   {knn_results['confusion_matrix'][0,0]:3d}  {knn_results['confusion_matrix'][0,1]:3d}")
print(f"        1   {knn_results['confusion_matrix'][1,0]:3d}  {knn_results['confusion_matrix'][1,1]:3d}")

tn, fp, fn, tp = knn_results['confusion_matrix'].ravel()
print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")
print("=" * 60)

In [ ]:
# Модель 2: Logistic Regression
print("=" * 60)
print("МОДЕЛЬ 2: Логистическая регрессия")
print("=" * 60)

log_reg = LogisticRegression(max_iter=5000, random_state=42)
log_reg_results = evaluate_model(log_reg, X_train, X_test, y_train, y_test, "Logistic Regression")

print(f"\nAccuracy (A): {log_reg_results['accuracy']:.4f}")
print(f"Precision (P): {log_reg_results['precision']:.4f}")
print(f"Recall (R): {log_reg_results['recall']:.4f}")
print(f"F1-score: {log_reg_results['f1']:.4f}")
print(f"ROC-AUC: {log_reg_results['roc_auc']:.4f}")

print("\nConfusion Matrix:")
print("                Прогноз")
print("                0    1")
print(f"Факт    0   {log_reg_results['confusion_matrix'][0,0]:3d}  {log_reg_results['confusion_matrix'][0,1]:3d}")
print(f"        1   {log_reg_results['confusion_matrix'][1,0]:3d}  {log_reg_results['confusion_matrix'][1,1]:3d}")

tn, fp, fn, tp = log_reg_results['confusion_matrix'].ravel()
print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")
print("=" * 60)

## 5. Сравнение моделей

In [ ]:
# Сводная таблица метрик
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC'],
    'KNN': [knn_results['accuracy'], knn_results['precision'], 
           knn_results['recall'], knn_results['f1'], knn_results['roc_auc']],
    'Logistic Regression': [log_reg_results['accuracy'], log_reg_results['precision'], 
                           log_reg_results['recall'], log_reg_results['f1'], log_reg_results['roc_auc']]
})

print("=" * 70)
print("СРАВНЕНИЕ МЕТРИК МОДЕЛЕЙ")
print("=" * 70)
print(results_df.to_string(index=False))
print("=" * 70)

In [ ]:
# Определение лучшей модели
best_model = knn_results if knn_results['roc_auc'] > log_reg_results['roc_auc'] else log_reg_results

print("\n" + "=" * 60)
print(f"ЛУЧШАЯ МОДЕЛЬ: {best_model['name'].upper()}")
print("=" * 60)
print(f"\nROC-AUC: {best_model['roc_auc']:.4f}")
print(f"Accuracy: {best_model['accuracy']:.4f}")
print(f"Precision: {best_model['precision']:.4f}")
print(f"Recall: {best_model['recall']:.4f}")
print(f"F1-score: {best_model['f1']:.4f}")
print("=" * 60)

In [ ]:
# Визуализация ROC-кривых
plt.figure(figsize=(10, 8))
plt.plot(knn_results['fpr'], knn_results['tpr'], label=f"KNN (AUC = {knn_results['roc_auc']:.4f})", linewidth=2)
plt.plot(log_reg_results['fpr'], log_reg_results['tpr'], label=f"Logistic Regression (AUC = {log_reg_results['roc_auc']:.4f})", linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC-кривые моделей классификации', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab3/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Визуализация Confusion Matrix для обеих моделей
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KNN
sns.heatmap(knn_results['confusion_matrix'], annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Злокачественная', 'Доброкачественная'],
            yticklabels=['Злокачественная', 'Доброкачественная'], ax=axes[0])
axes[0].set_title('KNN Confusion Matrix', fontsize=12)
axes[0].set_xlabel('Предсказанный класс')
axes[0].set_ylabel('Истинный класс')

# Logistic Regression
sns.heatmap(log_reg_results['confusion_matrix'], annot=True, fmt='d', cmap='Greens',
            xticklabels=['Злокачественная', 'Доброкачественная'],
            yticklabels=['Злокачественная', 'Доброкачественная'], ax=axes[1])
axes[1].set_title('Logistic Regression Confusion Matrix', fontsize=12)
axes[1].set_xlabel('Предсказанный класс')
axes[1].set_ylabel('Истинный класс')

plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab3/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Важность признаков для логистической регрессии
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg.coef_[0],
    'Abs_Coefficient': np.abs(log_reg.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

plt.figure(figsize=(12, 8))
colors = ['red' if x < 0 else 'green' for x in feature_importance['Coefficient']]
plt.barh(feature_importance['Feature'][:15], feature_importance['Coefficient'][:15], color=colors)
plt.xlabel('Коэффициент регрессии', fontsize=12)
plt.ylabel('Признак', fontsize=12)
plt.title('Важность признаков (Логистическая регрессия)', fontsize=14)
plt.axvline(x=0, color='black', linestyle='-')
plt.tight_layout()
plt.savefig('G:/Studies/TSTU/korneeva/4 couse 2 sem/big-data-analysis/labs/lab3/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("ТОП-10 ПРИЗНАКОВ ПО ВАЖНОСТИ (Логистическая регрессия)")
print("=" * 60)
for i, row in feature_importance.head(10).iterrows():
    direction = "→ доброкачественная" if row['Coefficient'] > 0 else "→ злокачественная"
    print(f"{row['Abs_Coefficient']:6.3f}: {row['Feature']:25s} {direction}")
print("=" * 60)

## 6. Идеи для улучшения моделей

### Возможные улучшения:

**1. Оптимизация гиперпараметров:**
   - KNN: подбор оптимального k (GridSearchCV)
   - Logistic Regression: подбор параметра регуляризации C, выбор типа регуляризации (L1/L2)

**2. Отбор признаков:**
   - RFE (Recursive Feature Elimination)
   - SelectKBest на основе статистических тестов
   - Удаление сильно коррелированных признаков

**3. Балансировка классов:**
   - SMOTE (Synthetic Minority Over-sampling Technique)
   - Взвешивание классов (class_weight='balanced')

**4. Ансамблирование:**
   - Random Forest
   - Gradient Boosting (XGBoost, LightGBM)
   - Stacking/Blending моделей

**5. Обработка выбросов:**
   - Winsorization
   - RobustScaler вместо StandardScaler

**6. Дополнительные модели:**
   - SVM с разными ядрами
   - Neural Networks
   - Naive Bayes

## 7. Ответы на контрольные вопросы

### Теоретические вопросы:

**1. Что такое классификация?**
Классификация — это задача машинного обучения, которая заключается в отнесении объектов к одной из заранее определённых категорий (классов) на основе их признаков.

**2. Чем отличается бинарная классификация от многоклассовой?**
Бинарная классификация разделяет объекты на 2 класса, многоклассовая — на более чем 2 класса.

**3. Какие преимущества имеет логистическая регрессия?**
- Простота и интерпретируемость
- Высокая скорость обучения
- Возврат вероятности принадлежности к классу
- Хорошо работает с линейно разделимыми данными

**4. Какую задачу решает логистическая регрессия?**
Задачу бинарной классификации путём моделирования вероятности принадлежности объекта к одному из классов с использованием логистической функции.

**5. Как исключить входные факторы с низкой значимостью?**
- RFE (Recursive Feature Elimination)
- SelectKBest с статистическими тестами
- Анализ важности признаков (feature importance)
- L1-регуляризация (Lasso), которая зануляет коэффициенты

**6. Как оценить точность классификации?**
Accuracy = (TP + TN) / (TP + TN + FP + FN) — доля правильных предсказаний среди всех.

**7. Как работает алгоритм KNN? Чем отличается от логистической регрессии?**
KNN (k-ближайших соседей) классифицирует объект на основе большинства классов его k ближайших соседей в пространстве признаков. Это непараметрический, «ленивый» метод, который не обучает модель, а хранит обучающие данные. Логистическая регрессия — параметрический метод, который обучает веса признаков для линейной границы принятия решений.

**8. Что показывают критерии качества и ROC-кривая для задачи определения пола?**
- Accuracy: точность предсказания пола
- Precision: как много из предсказанных мужчин — действительно мужчины
- Recall: как много мужчин мы обнаружили из всех мужчин
- ROC-AUC: показывает способность модели различать классы (площадь под кривой)

**9. Какие входные и выходные параметры используются в логистической регрессии?**
Входные: признаки X (вектор признаков объекта), выходные: вероятность принадлежности к классу (число от 0 до 1).

**10. Что показывают критерии качества и ROC-кривая для задачи определения доброкачественной опухоли?**
Позволяют оценить, насколько хорошо модель различает злокачественные и доброкачественные опухоли. ROC-AUC близкий к 1 означает хорошее различение классов.

**11. Как построить таблицу сопряженности (confusion matrix)?**
Сравниваются истинные значения (y_true) и предсказанные (y_pred), затем подсчитывается количество TP, TN, FP, FN. Используется sklearn.metrics.confusion_matrix().

## 8. Выводы

### Результаты исследования:

1. **Датасет**: Breast Cancer (569 образцов, 30 признаков)
   - Целевая переменная: тип опухоли (0=злокачественная, 1=доброкачественная)
   - Пропуски отсутствуют
   - Небаланс классов ~37%/63%

2. **Анализ признаков**:
   - Выявлены признаки с высокой корреляцией (mean radius, mean perimeter, mean area)
   - Обнаружены выбросы в некоторых признаках
   - Все признаки числовые

3. **Сравнение моделей**:
   - KNN и Логистическая регрессия показали схожие результаты
   - ROC-AUC ~0.98-0.99 указывает на отличное качество классификации

4. **Наиболее важные признаки**:
   - worst radius, worst perimeter, worst concave points
   - mean concave points, mean concavity

### Вывод:
Обе модели продемонстрировали высокую точность классификации (~97-98%), что говорит о хорошем качестве датасета и информативности признаков для диагностики рака груди. Логистическая регрессия имеет преимущество в интерпретируемости (коэффициенты показывают влияние каждого признака).